# Downloading the "BGLab/MaizeField3D" dataset

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="BGLab/MaizeField3D",
    repo_type="dataset",
    local_dir="/content/maize_dataset"
)

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

'/content/maize_dataset'

#Unzipping zipped file

In [ ]:
import zipfile
import os

zip_path = "/content/maize_dataset/datasets/FielGrwon_ZeaMays_RawPCD_100k.zip"
extract_path = "/content/raw_100k"

with zipfile.ZipFile(zip_path,'r') as zip_ref:
  zip_ref.extractall(extract_path)
print("Unzipped")

Unzipped


In [ ]:
!pip install open3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 133.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 84.6 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

Loading a .ply file

In [ ]:
pcd = o3d.io.read_point_cloud("/content/raw_100k/FielGrwon_ZeaMays_RawPCD_100k/0001.ply")

points = np.asarray(pcd.points)

print(points.shape)

(100000, 3)


# PointNet Encoder

In [ ]:
class PointEncoder(nn.Module):
  def __init__(self, latent_dim=128):
    super().__init__()

    self.mlp = nn.Sequential(
        nn.Linear(3,64),
        nn.ReLU(),
        nn.Linear(64,128),
        nn.ReLU(),
        nn.Linear(128, latent_dim)
    )
  def forward(self, x):
    x = self.mlp(x)
    x = torch.max(x, dim=1)[0]  # global max pooling
    return x

This encoder extracts gloabl shape features.<br>
max pooling --> permutation invariant

# Decoder

In [ ]:
class PointDecoder(nn.Module):
    def __init__(self, latent_dim=128, num_points=1024):
        super().__init__()
        self.num_points = num_points
        self.fc = nn.Sequential(
            nn.Linear(latent_dim,256),
            nn.ReLU(),
            nn.Linear(256,512),
            nn.ReLU(),
            nn.Linear(512, num_points*3)
        )
    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, self.num_points, 3)
        return x

# Full Autoencoder

In [ ]:
class PointAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = PointEncoder()
        self.decoder = PointDecoder()
    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon

### Normalizing the point cloud

In [ ]:
# points is numpy array from .ply
points = np.asarray(pcd.points)

# Downsample to 1024 points
def sample_points(points, n=1024):
    idx = np.random.choice(points.shape[0], n, replace=False)
    return points[idx]

points = sample_points(points, 1024)

# Normalize
points = points - np.mean(points, axis=0)
points = points / np.max(np.linalg.norm(points, axis=1))

###  Converting to tensor

In [ ]:
points = torch.tensor(points, dtype=torch.float32)
points = points.unsqueeze(0)  # batch dimension

device = "cuda" if torch.cuda.is_available() else "cpu"

points = points.to(device)

In [ ]:
print(points.shape)

torch.Size([1, 1024, 3])


#Chamfer Distance Loss

Chamfer Loss (Chamfer Distance) is a metric used to measure the similarity between two point clouds.

Since point clouds are unordered sets of points, we cannot compare them with normal losses like MSE directly.

Chamfer loss solves this by measuring how close every point in one set is to the other set.

Two terms:

1️⃣ Distance from original → reconstructed

2️⃣ Distance from reconstructed → original

In [ ]:
def chamfer_distance(pc1, pc2):

    diff = pc1.unsqueeze(2) - pc2.unsqueeze(1)

    dist = torch.sum(diff**2, dim=-1)

    loss1 = torch.min(dist, dim=2)[0]
    loss2 = torch.min(dist, dim=1)[0]

    return torch.mean(loss1) + torch.mean(loss2)

In [ ]:
model = PointAutoencoder().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 200

for epoch in range(epochs):

    model.train()

    recon = model(points)

    loss = chamfer_distance(points, recon)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 40 == 0:
        print(f"Epoch {epoch+1} | Loss {loss.item():.6f}")

Epoch 40 | Loss 0.000684
Epoch 80 | Loss 0.000364
Epoch 120 | Loss 0.000357
Epoch 160 | Loss 0.000357
Epoch 200 | Loss 0.000357


In [ ]:
model.eval()
with torch.no_grad():
    recon = model(points)

#Convert to numpy
recon = recon.squeeze(0).cpu().numpy()
original = points.squeeze(0).cpu().numpy()

In [ ]:
!pip install plotly

# Visualizing Original Point Cloud

In [ ]:
import plotly.graph_objects as go
import numpy as np
fig = go.Figure(data=[go.Scatter3d(
    x=original[:,0],
    y=original[:,1],
    z=original[:,2],
    mode='markers',
    marker=dict(
        size=2,
        color='green'
    )
)])

fig.update_layout(
    title="Original Point Cloud",
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    )
)

fig.show()

# Visualizing Reconstructed Point Cloud

In [ ]:
fig = go.Figure(data=[go.Scatter3d(
    x=recon[:,0],
    y=recon[:,1],
    z=recon[:,2],
    mode='markers',
    marker=dict(
        size=2,
        color='red'
    )
)])

fig.update_layout(
    title="Reconstructed Point Cloud"
)

fig.show()

#Original vs Reconstruction

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=original[:,0],
    y=original[:,1],
    z=original[:,2],
    mode='markers',
    marker=dict(size=2, color='green', opacity=0.6),
    name="Original"
))

fig.add_trace(go.Scatter3d(
    x=recon[:,0],
    y=recon[:,1],
    z=recon[:,2],
    mode='markers',
    marker=dict(size=2, color='red'),
    name="Reconstructed"
))

fig.update_layout(
    title="Original vs Reconstructed Point Cloud",
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    )
)

fig.show()

### PointNet’s big trick is max pooling. The network extracts the most important feature across all points. This makes the model order invariant

In [ ]:
!pip install scikit-learn

# Autoencoder for all 1045 .ply files

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import open3d as o3d
from torch.utils.data import Dataset, DataLoader
import plotly.graph_objects as go
from sklearn.decomposition import PCA

100k points --> 1024 points

In [ ]:
def sample_points(points, n=1024):
    idx = np.random.choice(points.shape[0], n, replace=False)
    return points[idx]

In [ ]:
def normalize_points(points):

    points = points - np.mean(points, axis=0)
    scale = np.max(np.linalg.norm(points, axis=1))
    points = points / scale

    return points

In [ ]:
class MaizeDataset(Dataset):

    def __init__(self, root_dir, num_points=1024):

        self.root_dir = root_dir
        self.files = [f for f in os.listdir(root_dir) if f.endswith(".ply")]
        self.num_points = num_points

    def __len__(self):

        return len(self.files)

    def __getitem__(self, idx):

        file_path = os.path.join(self.root_dir, self.files[idx])
        pcd = o3d.io.read_point_cloud(file_path)
        points = np.asarray(pcd.points)
        points = sample_points(points, self.num_points)
        points = normalize_points(points)
        points = torch.tensor(points, dtype=torch.float32)

        return points

In [ ]:
data_path = "/content/raw_100k/FielGrwon_ZeaMays_RawPCD_100k"

dataset = MaizeDataset(data_path)

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2
)
batch = next(iter(loader))
print(batch.shape)

torch.Size([16, 1024, 3])


In [ ]:
def chamfer_distance(pc1, pc2):

    diff = pc1.unsqueeze(2) - pc2.unsqueeze(1)
    dist = torch.sum(diff**2, dim=-1)
    loss1 = torch.min(dist, dim=2)[0]
    loss2 = torch.min(dist, dim=1)[0]

    return torch.mean(loss1) + torch.mean(loss2)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = PointAutoencoder().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs = 50

for epoch in range(epochs):

    total_loss = 0
    for points in loader:
        points = points.to(device)
        recon = model(points)
        loss = chamfer_distance(points, recon)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch} | Loss {total_loss/len(loader):.6f}")

Epoch 0 | Loss 0.022256
Epoch 1 | Loss 0.013223
Epoch 2 | Loss 0.012643
Epoch 3 | Loss 0.010003
Epoch 4 | Loss 0.009053
Epoch 5 | Loss 0.008662
Epoch 6 | Loss 0.007838
Epoch 7 | Loss 0.007117
Epoch 8 | Loss 0.006714
Epoch 9 | Loss 0.006466
Epoch 10 | Loss 0.006249
Epoch 11 | Loss 0.006054
Epoch 12 | Loss 0.005811
Epoch 13 | Loss 0.005600
Epoch 14 | Loss 0.005390
Epoch 15 | Loss 0.005307
Epoch 16 | Loss 0.005227
Epoch 17 | Loss 0.005028
Epoch 18 | Loss 0.004977
Epoch 19 | Loss 0.004881
Epoch 20 | Loss 0.004753
Epoch 21 | Loss 0.004701
Epoch 22 | Loss 0.004605
Epoch 23 | Loss 0.004479
Epoch 24 | Loss 0.004415
Epoch 25 | Loss 0.004376
Epoch 26 | Loss 0.004302
Epoch 27 | Loss 0.004169
Epoch 28 | Loss 0.004136
Epoch 29 | Loss 0.004042
Epoch 30 | Loss 0.003936
Epoch 31 | Loss 0.003905
Epoch 32 | Loss 0.003883
Epoch 33 | Loss 0.003811
Epoch 34 | Loss 0.003785
Epoch 35 | Loss 0.003705
Epoch 36 | Loss 0.003622
Epoch 37 | Loss 0.003599
Epoch 38 | Loss 0.003554
Epoch 39 | Loss 0.003474
Epoch 40 |

#Visualizing Reconstruction

In [ ]:
model.eval()

sample = dataset[0].unsqueeze(0).to(device)

with torch.no_grad():

    recon = model(sample)

original = sample.squeeze(0).cpu().numpy()
recon = recon.squeeze(0).cpu().numpy()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=original[:,0],
    y=original[:,1],
    z=original[:,2],
    mode='markers',
    marker=dict(size=2,color='green'),
    name='Original'
))

fig.add_trace(go.Scatter3d(
    x=recon[:,0],
    y=recon[:,1],
    z=recon[:,2],
    mode='markers',
    marker=dict(size=2,color='red'),
    name='Reconstructed'
))

fig.update_layout(title="Plant Reconstruction")

fig.show()

# Visualizing the latent space for all the .ply files

Computing latent vectors for all plants

In [ ]:
latents = []
model.eval()

with torch.no_grad():

    for points in loader:

        points = points.to(device)

        z = model.encoder(points)

        latents.append(z.cpu().numpy())

latents = np.concatenate(latents)

In [ ]:
print(latents.shape)

(1045, 128)


Reducing latent space to 2d using PCA

In [ ]:
pca = PCA(n_components=2)

latent_2d = pca.fit_transform(latents)

In [ ]:
fig = go.Figure(data=[go.Scatter(
    x=latent_2d[:,0],
    y=latent_2d[:,1],
    mode='markers'
)])

fig.update_layout(
    title="Maize Plant Latent Space",
    xaxis_title="Latent Dim 1",
    yaxis_title="Latent Dim 2"
)

fig.show()

What does it represents:
-  1 dot = 1 maize plant<br>
- Plant point cloud-->
PointNet Encoder-->
128-D latent vector-->
PCA-->
2-D visualization <br>

So the figure shows how 1045 plants are arranged in feature space.

Plants with similar geometry should appear closer together.

# Final pipeline overview:


Maize Point Cloud<br>
        ↓<br>
PointNet Encoder<br>
        ↓<br>
128-dim plant representation<br>
        ↓<br>
PointNet Decoder<br>
        ↓<br>
Reconstructed plant